In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Hardware: Colab T4 — runs in ~15-30 minutes
# =============================================================================
 
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from transformers import ViTModel, ViTConfig
import math
import time
 
# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")


from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

In [ ]:
# =============================================================================
# HYPERPARAMETERS — from the paper's Table 4
# =============================================================================
 
IMAGE_SIZE   = 224       # paper uses 384, but 224 fits on T4
NUM_CLASSES  = 10        # CIFAR-10
BATCH_SIZE   = 64       # paper uses 512, but 128 fits on T4
TOTAL_STEPS  = 10000     # paper: 10,000 for CIFAR
WARMUP_STEPS = 500       # linear warmup
BASE_LR      = 0.01      # from paper's grid {0.001, 0.003, 0.01, 0.03}
MOMENTUM     = 0.9       # paper: SGD with momentum 0.9
GRAD_CLIP    = 1.0       # paper: grad clipping at global norm 1
 
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
# =============================================================================
# STEP 1 — LOAD CIFAR-10 WITH IMAGENET NORMALIZATION
#
# Key point: we normalize with ImageNet stats, not CIFAR-10 stats.
# Why? Because the ViT was PRE-TRAINED on ImageNet. Its weights expect
# inputs normalized with ImageNet's mean/std. If we used CIFAR-10 stats,
# the pixel values would be slightly "off" from what the model learned.
#
# How to verify these numbers: check the model's preprocessor config at
# https://huggingface.co/google/vit-base-patch16-224-in21k
# The config file (preprocessor_config.json) lists the exact values.
# =============================================================================
 
print("\n" + "="*60)
print("STEP 1: Loading CIFAR-10 with ImageNet normalization")
print("="*60)
 
# ImageNet stats — these come from the model's preprocessor config on HuggingFace
# NOT from the ViT paper itself (the paper doesn't specify them)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
 
# Training transforms: resize + flip + normalize
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),    # 32×32 → 224×224
    transforms.RandomHorizontalFlip(),               # standard augmentation
    transforms.ToTensor(),                            # [0,255] uint8 → [0,1] float
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD) # ImageNet normalization
])
 
# Test transforms: resize + normalize only (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
 
# Download CIFAR-10
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform
)
 
# DataLoaders — handle batching, shuffling, parallel loading
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True    # pin_memory speeds up GPU transfer
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)
 
# Quick check
sample_images, sample_labels = next(iter(train_loader))
print(f"Batch shape:  {sample_images.shape}")     # (128, 3, 224, 224)
print(f"Labels shape: {sample_labels.shape}")      # (128,)
print(f"Pixel range:  [{sample_images.min():.2f}, {sample_images.max():.2f}]")
 
steps_per_epoch = len(train_loader)
total_epochs    = math.ceil(TOTAL_STEPS / steps_per_epoch)
print(f"Steps/epoch:  {steps_per_epoch}")
print(f"Total epochs: {total_epochs}")
 

In [ ]:
# =============================================================================
 
print("\n" + "="*60)
print("STEP 2: Loading pre-trained ViT-Base from HuggingFace")
print("="*60)
 
# This downloads ~330MB of pre-trained weights the first time
# After that, it's cached locally
pretrained_encoder = ViTModel.from_pretrained(
    'google/vit-base-patch16-224-in21k'
)
 
print(f"Pre-trained encoder loaded!")
print(f"Hidden size:   {pretrained_encoder.config.hidden_size}")
print(f"Num layers:    {pretrained_encoder.config.num_hidden_layers}")
print(f"Num heads:     {pretrained_encoder.config.num_attention_heads}")
print(f"Patch size:    {pretrained_encoder.config.patch_size}")
 
# Count pre-trained parameters
pretrained_params = sum(p.numel() for p in pretrained_encoder.parameters())
print(f"Parameters:    {pretrained_params:,}")

In [ ]:
# =============================================================================
# STEP 3 — BUILD THE FINE-TUNING MODEL
#
# Architecture:
#   [Pre-trained ViT encoder] → CLS token (768-dim) → [New linear head (10)]
#
# The encoder is everything we built line-by-line before:
#   patch embedding → CLS token → pos embeddings → 12 transformer blocks → LN
#
# We just add a single Dense(768 → 10) on top.
#
# Paper (B.1.1): "replace it by a single, zero-initialized linear layer"
# =============================================================================
 
print("\n" + "="*60)
print("STEP 3: Building fine-tuning model (encoder + new head)")
print("="*60)
 
 
class ViTForCIFAR10(nn.Module):
    """
    Pre-trained ViT encoder + new classification head.
 
    The encoder does:
      image → patches → [CLS] + patch embeddings + pos embeddings
      → 12 transformer blocks → LayerNorm → CLS token output (768-dim)
 
    We add:
      CLS output (768) → Linear(768, 10) → logits
    """
 
    def __init__(self, pretrained_encoder, num_classes):
        super().__init__()
 
        # The pre-trained encoder — all 86M parameters, already trained
        self.encoder = pretrained_encoder
 
        # New classification head — zero-initialized per the paper
        self.head = nn.Linear(
            pretrained_encoder.config.hidden_size,   # 768
            num_classes                               # 10
        )
 
        # Zero-initialize the head (paper's recommendation)
        # Why? With zero weights, all logits start at 0
        # → softmax gives uniform 1/10 probability per class
        # → initial loss = -log(0.1) = 2.3026 (cross-entropy of random guess)
        # This ensures the first gradient updates are driven entirely by
        # the pre-trained features, not random head weights.
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)
 
    def forward(self, pixel_values):
        # Forward through the pre-trained encoder
        # This runs the ENTIRE pipeline we built line-by-line:
        #   Conv2D patch projection → CLS token → pos embed → 12 blocks → LN
        outputs = self.encoder(pixel_values=pixel_values)
 
        # outputs.last_hidden_state shape: (B, 197, 768) — all token outputs
        # outputs.last_hidden_state[:, 0] — the CLS token at position 0
        # This is z_0^L from paper eq. 4: the image summary vector
        cls_output = outputs.last_hidden_state[:, 0]   # (B, 768)
 
        # Classification head
        logits = self.head(cls_output)                  # (B, 10)
        return logits
 
 
# Build the model
model = ViTForCIFAR10(pretrained_encoder, NUM_CLASSES)
model = model.to(device)
 
# Verify the zero-initialized head
with torch.no_grad():
    test_input = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
    test_logits = model(test_input)
    print(f"Initial logits (should be ~0): {test_logits.cpu().numpy().round(4)}")
    test_probs = torch.softmax(test_logits, dim=-1)
    print(f"Initial probs  (should be ~0.1 each): {test_probs.cpu().numpy().round(4)}")
 
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# =============================================================================
# STEP 4 — LEARNING RATE SCHEDULE: COSINE DECAY WITH WARMUP
#
# Same as before — linear warmup then cosine decay.
# Paper (Table 4): "All models are fine-tuned with cosine learning rate decay"
#
#   LR
#   ^
#   |     /‾‾‾‾‾‾\
#   |    /          \
#   |   /            \
#   |  /              \___
#   +-----|------------|----→ step
#     warmup    cosine decay
# =============================================================================
 
print("\n" + "="*60)
print("STEP 4: Setting up cosine LR schedule + SGD optimizer")
print("="*60)
 
 
def get_lr(step):
    """Compute learning rate for a given step (warmup + cosine decay)."""
    if step < WARMUP_STEPS:
        # Linear warmup: 0 → BASE_LR
        return BASE_LR * (step / WARMUP_STEPS)
    else:
        # Cosine decay: BASE_LR → 0
        progress = (step - WARMUP_STEPS) / (TOTAL_STEPS - WARMUP_STEPS)
        return 0.5 * BASE_LR * (1.0 + math.cos(math.pi * progress))
 
 
# Show LR at key points
for s in [0, 250, 500, 2500, 5000, 7500, 10000]:
    print(f"  Step {s:>6d}: LR = {get_lr(s):.6f}")
 
 
# SGD with momentum — paper's choice for fine-tuning
# No weight decay — paper's Table 4 says weight_decay = 0
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=BASE_LR,
    momentum=MOMENTUM    # 0.9
)
 
print(f"\nOptimizer: SGD(lr={BASE_LR}, momentum={MOMENTUM})")
print(f"Grad clip: global norm {GRAD_CLIP}")

In [ ]:
# =============================================================================
# STEP 5 — THE TRAINING LOOP (FINE-TUNING) — with tqdm progress bars
# =============================================================================

from tqdm.auto import tqdm

print("\n" + "="*60)
print("STEP 5: FINE-TUNING BEGINS")
print("="*60)

loss_fn = nn.CrossEntropyLoss()

global_step = 0
best_test_acc = 0.0
start_time = time.time()

model.train()

for epoch in range(total_epochs):

    epoch_loss = 0.0
    epoch_correct = 0
    epoch_total = 0

    # tqdm wraps the dataloader — gives you a live progress bar
    pbar = tqdm(train_loader,
                desc=f"Epoch {epoch+1}/{total_epochs}",
                leave=True)

    for batch_images, batch_labels in pbar:

        if global_step >= TOTAL_STEPS:
            break

        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        # Update learning rate
        current_lr = get_lr(global_step)
        for param_group in optimizer.param_groups:
            param_group['lr'] = current_lr

        # --- The 6 training lines (unchanged) ---
        logits = model(batch_images)
        loss = loss_fn(logits, batch_labels)
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        optimizer.zero_grad()

        # Track metrics
        epoch_loss += loss.item()
        predictions = logits.argmax(dim=-1)
        epoch_correct += (predictions == batch_labels).sum().item()
        epoch_total += batch_labels.size(0)
        global_step += 1

        # Update the progress bar with live metrics
        avg_loss = epoch_loss / (epoch_total / BATCH_SIZE)
        avg_acc  = epoch_correct / epoch_total
        pbar.set_postfix({
            'loss': f'{avg_loss:.4f}',
            'acc': f'{avg_acc:.4f}',
            'lr': f'{current_lr:.5f}',
            'step': f'{global_step}/{TOTAL_STEPS}'
        })

    pbar.close()

    if global_step >= TOTAL_STEPS:
        break

    # --- Evaluate on test set ---
    model.eval()
    test_correct = 0
    test_total = 0

    test_pbar = tqdm(test_loader, desc="  Evaluating", leave=False)
    with torch.no_grad():
        for test_imgs, test_lbls in test_pbar:
            test_imgs = test_imgs.to(device)
            test_lbls = test_lbls.to(device)
            test_logits = model(test_imgs)
            test_preds = test_logits.argmax(dim=-1)
            test_correct += (test_preds == test_lbls).sum().item()
            test_total += test_lbls.size(0)
    test_pbar.close()

    test_acc = test_correct / test_total
    train_acc = epoch_correct / epoch_total if epoch_total > 0 else 0
    if test_acc > best_test_acc:
        best_test_acc = test_acc

    elapsed = time.time() - start_time
    print(f"  ✓ Train acc: {train_acc:.4f} | "
          f"Test acc: {test_acc:.4f} | "
          f"Best: {best_test_acc:.4f} | "
          f"Time: {elapsed:.0f}s\n")

    model.train()

In [ ]:
# =============================================================================
# STEP 6 — FINAL EVALUATION
# =============================================================================
 
print("\n" + "="*60)
print("STEP 6: Final evaluation")
print("="*60)
 
model.eval()
test_correct = 0
test_total = 0
class_correct = [0] * NUM_CLASSES
class_total   = [0] * NUM_CLASSES
 
with torch.no_grad():
    for test_imgs, test_lbls in test_loader:
        test_imgs = test_imgs.to(device)
        test_lbls = test_lbls.to(device)
        test_logits = model(test_imgs)
        test_preds = test_logits.argmax(dim=-1)
 
        test_correct += (test_preds == test_lbls).sum().item()
        test_total += test_lbls.size(0)
 
        # Per-class tracking
        for i in range(len(test_lbls)):
            label = test_lbls[i].item()
            class_total[label] += 1
            if test_preds[i].item() == label:
                class_correct[label] += 1
 
final_acc = test_correct / test_total
total_time = time.time() - start_time
 
print(f"Final test accuracy: {final_acc:.4f} ({final_acc*100:.1f}%)")
print(f"Total training time: {total_time:.0f}s ({total_time/60:.1f} minutes)")
print(f"\nPer-class accuracy:")
for i, name in enumerate(CLASS_NAMES):
    if class_total[i] > 0:
        acc = class_correct[i] / class_total[i]
        print(f"  {name:>12s}: {acc:.4f} ({acc*100:.1f}%)")
 
 
# =============================================================================
# STEP 7 — SAVE THE FINE-TUNED MODEL
# =============================================================================
 
print("\n" + "="*60)
print("STEP 7: Saving model")
print("="*60)
 
# Save just the model weights
torch.save(model.state_dict(), 'vit_base_cifar10_finetuned.pth')
print("Saved to: vit_base_cifar10_finetuned.pth")
 
# To reload later:
print("\nTo reload:")
print("  model = ViTForCIFAR10(ViTModel.from_pretrained('google/vit-base-patch16-224-in21k'), 10)")
print("  model.load_state_dict(torch.load('vit_base_cifar10_finetuned.pth'))")